In [ ]:
# Data Ingestion, Cleaning & Preprocessing with Pandas

## Task 03 — Retail Store Sales Dataset

This notebook performs end-to-end data ingestion, data quality assessment,
cleaning, preprocessing, feature engineering, validation, and export of a
retail store sales dataset using Python and Pandas.

### Objectives

- Load and inspect the raw retail sales dataset.
- Identify missing values and duplicate records.
- Detect and correct inconsistent data types.
- Handle invalid and extreme values.
- Standardize categorical and numerical fields.
- Perform feature engineering using transaction dates and sales values.
- Compare the dataset before and after cleaning.
- Export the final standardized dataset as `clean_dataset.csv`.

### Tools Used

- Python
- Pandas
- NumPy
- Matplotlib
- Jupyter Notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

In [ ]:
file_path = "data/retail_store_sales.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")

In [ ]:
raw_df = df.copy()

print("Working copy created.")

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nDataset shape:", df.shape)

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
print("Columns in the dataset:\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
missing_before = df.isnull().sum()

missing_before = missing_before[missing_before > 0].sort_values(ascending=False)

print("Missing values before cleaning:\n")
print(missing_before)

In [ ]:
missing_percentage = (
    df.isnull().mean() * 100
).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": missing_percentage.round(2)
})

missing_summary[missing_summary["Missing_Count"] > 0]

In [ ]:
missing_plot = df.isnull().sum()

missing_plot = missing_plot[missing_plot > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
missing_plot.plot(kind="bar")
plt.title("Missing Values Before Cleaning")
plt.xlabel("Columns")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

In [ ]:
duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

In [ ]:
dtype_summary = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values
})

dtype_summary

In [ ]:
for column in df.columns:
    print(f"\n{'=' * 60}")
    print(f"Column: {column}")
    print("Unique values:", df[column].nunique(dropna=True))
    
    if df[column].nunique(dropna=True) <= 20:
        print(df[column].dropna().unique())

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

print("Numeric columns:")
print(numeric_columns)

In [ ]:
numeric_columns = [
    "Price Per Unit",
    "Quantity",
    "Total Spent"
]

for column in numeric_columns:
    if column in df.columns:
        negative_count = (df[column] < 0).sum()
        print(f"{column}: {negative_count} negative values")

In [ ]:
print(df["Transaction Date"].head(20))
print("\nData type:", df["Transaction Date"].dtype)

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(df.columns.tolist())

In [ ]:
for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].str.strip()

print("Whitespace cleaned from text columns.")

In [ ]:
text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = df[column].apply(
        lambda x: x.title() if isinstance(x, str) else x
    )

print("Categorical text standardized.")

In [ ]:
numeric_columns = [
    "price_per_unit",
    "quantity",
    "total_spent"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Numeric columns converted successfully.")

In [ ]:
def clean_discount(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip().lower()
    
    if value in ["yes", "y", "true", "1"]:
        return 1
    
    if value in ["no", "n", "false", "0"]:
        return 0
    
    return np.nan


df["discount_applied"] = df["discount_applied"].apply(clean_discount)

print(df["discount_applied"].value_counts(dropna=False))

In [ ]:
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    errors="coerce"
)

print(df["transaction_date"].dtype)

In [ ]:
invalid_dates = df["transaction_date"].isna().sum()

print("Invalid or missing dates:", invalid_dates)

In [ ]:
duplicates_before = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

duplicates_after = df.duplicated().sum()

print("Duplicates before:", duplicates_before)
print("Duplicates after:", duplicates_after)

In [ ]:
df["item"] = df["item"].fillna("Unknown")

print("Missing Item values after cleaning:",
      df["item"].isna().sum())

In [ ]:
df["category"] = df["category"].fillna("Unknown")

print("Missing Category values:",
      df["category"].isna().sum())

In [ ]:
df["payment_method"] = df["payment_method"].fillna("Unknown")

print("Missing Payment Method values:",
      df["payment_method"].isna().sum())

In [ ]:
df["discount_applied"] = df["discount_applied"].fillna(0).astype(int)

print(df["discount_applied"].value_counts())

In [ ]:
for column in ["price_per_unit", "quantity", "total_spent"]:
    df.loc[df[column] < 0, column] = np.nan

print("Negative numeric values converted to missing values.")

In [ ]:
df["price_per_unit"] = (
    df.groupby("category")["price_per_unit"]
      .transform(lambda x: x.fillna(x.median()))
)

df["price_per_unit"] = df["price_per_unit"].fillna(
    df["price_per_unit"].median()
)

print("Missing Price Per Unit:",
      df["price_per_unit"].isna().sum())

In [ ]:
df["quantity"] = (
    df.groupby("category")["quantity"]
      .transform(lambda x: x.fillna(x.median()))
)

df["quantity"] = df["quantity"].fillna(
    df["quantity"].median()
)

df["quantity"] = df["quantity"].round().astype(int)

print("Missing Quantity:",
      df["quantity"].isna().sum())

In [ ]:
calculated_total = (
    df["price_per_unit"] * df["quantity"]
)

df["total_spent"] = df["total_spent"].fillna(calculated_total)

print("Missing Total Spent:",
      df["total_spent"].isna().sum())

In [ ]:
expected_total = df["price_per_unit"] * df["quantity"]

difference = (df["total_spent"] - expected_total).abs()

print("Maximum difference:", difference.max())
print("Rows with significant difference:", (difference > 0.01).sum())

In [ ]:
if df["transaction_date"].isna().sum() > 0:
    mode_date = df["transaction_date"].mode()[0]
    df["transaction_date"] = df["transaction_date"].fillna(mode_date)

print("Missing dates after treatment:",
      df["transaction_date"].isna().sum())

In [ ]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]
    
    return outliers, lower_bound, upper_bound


for column in ["price_per_unit", "quantity", "total_spent"]:
    outliers, lower, upper = detect_outliers_iqr(df, column)
    
    print(f"\n{column}")
    print("Lower bound:", round(lower, 2))
    print("Upper bound:", round(upper, 2))
    print("Outliers:", len(outliers))

In [ ]:
def cap_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    data[column] = data[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )


for column in ["price_per_unit", "quantity"]:
    cap_outliers(df, column)

print("Outliers capped using IQR boundaries.")

In [ ]:
df["total_spent"] = (
    df["price_per_unit"] * df["quantity"]
)

df["total_spent"] = df["total_spent"].round(2)

print("Total Spent recalculated.")

In [ ]:
df["year"] = df["transaction_date"].dt.year

df[["transaction_date", "year"]].head()

In [ ]:
df["month"] = df["transaction_date"].dt.month

df[["transaction_date", "month"]].head()

In [ ]:
df["month_name"] = df["transaction_date"].dt.month_name()

df[["transaction_date", "month_name"]].head()

In [ ]:
df["average_transaction_value"] = (
    df["total_spent"] / df["quantity"]
).round(2)

df[[
    "total_spent",
    "quantity",
    "average_transaction_value"
]].head()

In [ ]:
df["discount_status"] = df["discount_applied"].map({
    0: "No Discount",
    1: "Discount Applied"
})

df[[
    "discount_applied",
    "discount_status"
]].head()

In [ ]:
print("Final columns:\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
missing_after = df.isnull().sum()

missing_after = missing_after[missing_after > 0]

print("Missing values after cleaning:")

if len(missing_after) == 0:
    print("No missing values remain.")
else:
    print(missing_after)

In [ ]:
final_duplicates = df.duplicated().sum()

print("Duplicate rows after cleaning:", final_duplicates)

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Missing Values"
    ],
    "Before Cleaning": [
        raw_df.shape[0],
        raw_df.shape[1],
        raw_df.duplicated().sum(),
        raw_df.isnull().sum().sum()
    ],
    "After Cleaning": [
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        df.isnull().sum().sum()
    ]
})

comparison

In [ ]:
missing_comparison = pd.DataFrame({
    "Before": raw_df.isnull().sum(),
    "After": df.isnull().sum()
})

missing_comparison["Reduction"] = (
    missing_comparison["Before"] -
    missing_comparison["After"]
)

missing_comparison

In [ ]:
plot_data = missing_comparison[
    ["Before", "After"]
].sort_values("Before", ascending=False)

plot_data.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Missing Values Before vs After Cleaning")
plt.xlabel("Columns")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

raw_df["Total Spent"].dropna().plot(
    kind="hist",
    bins=30
)

plt.title("Total Spent Distribution Before Cleaning")
plt.xlabel("Total Spent")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

df["total_spent"].plot(
    kind="hist",
    bins=30
)

plt.title("Total Spent Distribution After Cleaning")
plt.xlabel("Total Spent")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
df.head(10)

In [ ]:
validation_results = {
    "Rows >= 10,000": len(df) >= 10000,
    "No duplicate rows": df.duplicated().sum() == 0,
    "No missing values": df.isnull().sum().sum() == 0,
    "Transaction date is datetime": pd.api.types.is_datetime64_any_dtype(
        df["transaction_date"]
    ),
    "Quantity is numeric": pd.api.types.is_numeric_dtype(
        df["quantity"]
    ),
    "Price is numeric": pd.api.types.is_numeric_dtype(
        df["price_per_unit"]
    ),
    "Total Spent is numeric": pd.api.types.is_numeric_dtype(
        df["total_spent"]
    )
}

validation_df = pd.DataFrame(
    list(validation_results.items()),
    columns=["Validation Check", "Passed"]
)

validation_df

In [ ]:
output_path = "clean_dataset.csv"

df.to_csv(
    output_path,
    index=False
)

print(f"Clean dataset exported successfully to: {output_path}")

In [ ]:
clean_check = pd.read_csv(output_path)

print("Exported dataset shape:", clean_check.shape)

print("\nMissing values:")
print(clean_check.isnull().sum().sum())

print("\nDuplicate rows:")
print(clean_check.duplicated().sum())

In [ ]:
## Conclusion

The retail sales dataset was successfully ingested, profiled, cleaned,
standardized, and transformed using Pandas.

### Cleaning activities performed

- Inspected the raw dataset and its structure.
- Identified missing values and duplicate records.
- Standardized column names and text values.
- Converted numerical fields to appropriate data types.
- Converted transaction dates to datetime format.
- Standardized the discount indicator.
- Handled missing categorical values.
- Imputed missing numerical values using appropriate strategies.
- Recalculated Total Spent where required.
- Detected and capped extreme numerical values using the IQR method.
- Created year, month, and month-name features.
- Created Average Transaction Value.
- Created a Discount Status feature.
- Performed final data-quality validation.
- Exported the cleaned dataset as `clean_dataset.csv`.

The resulting dataset is standardized and ready for further analysis,
visualization, and business intelligence workflows.